# 🎬 YouTube Shorts Generator — Google Colab (Tek Dosya)

**Kullanım:**
1. Bu defteri Colab'a yükleyin ve **Runtime → Run all** (defter tek başınadır; proje dosyaları `%%writefile` hücrelerine gömülüdür)
2. Altına yeni bir **Code** hücresi açıp konuyu girin:
   ```python
   generate("Konu başlığı")
   ```
3. Video `output/` klasöründe üretilir ve hücre içinde önizlenir; Files panelinden indirebilirsiniz.

> ⚠️ API anahtarları bu defterin içine gömülüdür — defteri kimseyle paylaşmayın.


In [ ]:
# Gerekli paketleri kur ve proje dizinlerini oluştur
# DİKKAT: Colab moviepy'yi 1.0.3 olarak ÖNCEDEN kurar; bu proje MoviePy 2.x API
# kullanır. Pini olmadan pip "already satisfied" der ve 1.x kalır.
# jedi: Colab taban ortamında ipython (7.34.0) jedi>=0.16 ister ve jedi YOKTUR;
# pip bu eksik bağımlılığı sahte bir 'dependency conflicts' ERROR'ı olarak basar.
!pip install -q "moviepy>=2.0.0" google-genai edge-tts python-dotenv requests "decorator>=5.1.0,<6.0.0" jedi
# Altyazı fontu: Colab VM'inde DejaVuSans-Bold her zaman bulunmayabilir (config.py
# matplotlib fontuyla da çözümler; bu satır ikinci güvence)
!apt-get install -yqq fonts-dejavu-core >/dev/null 2>&1 || true

import os
os.makedirs("pipeline", exist_ok=True)
os.makedirs("services", exist_ok=True)

# Kurulum doğrulaması: moviepy 1.x üst seviye export içermez → burada patlamalı
import moviepy
try:
    from moviepy import VideoFileClip  # MoviePy 2.x üst seviye import
except ImportError as exc:
    raise ImportError(
        "moviepy 2.x yüklenemedi (eski 1.0.3 cache'i?). "
        "Çözüm: Runtime > Restart runtime, sonra Run all."
    ) from exc
print(f"✓ moviepy {moviepy.__version__} + paketler hazır, dizinler oluşturuldu")


In [ ]:
import os

# --- API anahtarlari (Colab Secrets veya ortam degiskeni) ---
# Gercek degerler deftere gomulmez; Colab Secrets kullanilir.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
CLOUDFLARE_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID", "")
CLOUDFLARE_API_TOKEN = os.getenv("CLOUDFLARE_API_TOKEN", "")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["CLOUDFLARE_ACCOUNT_ID"] = CLOUDFLARE_ACCOUNT_ID
os.environ["CLOUDFLARE_API_TOKEN"] = CLOUDFLARE_API_TOKEN

print("GEMINI_API_KEY:", "SET (" + str(len(GEMINI_API_KEY)) + " ch)" if GEMINI_API_KEY else "MISSING!")
print("CLOUDFLARE_ACCOUNT_ID:", "SET (%d ch)" % len(CLOUDFLARE_ACCOUNT_ID))
print("CLOUDFLARE_API_TOKEN:", "SET (%d ch)" % len(CLOUDFLARE_API_TOKEN))
cf_ok = bool(CLOUDFLARE_ACCOUNT_ID) and bool(CLOUDFLARE_API_TOKEN)
print("CLOUDFLARE:", "SET" if cf_ok else "MISSING")

In [ ]:
%%writefile config.py
"""Project configuration — all tunables live here (see .env.example)."""
import os
import urllib.request
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Project paths
BASE_DIR = Path(__file__).parent
OUTPUT_DIR = BASE_DIR / "output"
TMP_DIR = OUTPUT_DIR / "tmp"
IMAGES_DIR = TMP_DIR / "images"
AUDIO_DIR = TMP_DIR / "audio"

# Ensure directories exist
OUTPUT_DIR.mkdir(exist_ok=True)
TMP_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
AUDIO_DIR.mkdir(exist_ok=True)

# API Keys
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# Video settings
VIDEO_WIDTH = 1080
VIDEO_HEIGHT = 1920
MIN_DURATION = 40  # seconds
MAX_DURATION = 60  # seconds

# Scene settings
MIN_SCENES = 5
MAX_SCENES = 8

# Voice settings
VOICE_LANGUAGE = "tr-TR"  # Turkish
VOICE_NAME = "tr-TR-AhmetNeural"  # Male Turkish voice
# Edge TTS 403/throttle backoff: rapid sequential WSS connects get rate-limited
TTS_MAX_RETRIES = int(os.getenv("TTS_MAX_RETRIES", "5"))
TTS_RETRY_BASE_DELAY = float(os.getenv("TTS_RETRY_BASE_DELAY", "3.0"))  # s, exponential: base * 2**attempt
TTS_SCENE_DELAY = float(os.getenv("TTS_SCENE_DELAY", "1.5"))  # s between scenes

# Subtitle settings - TikTok style (bottom, scroll-up animation)
_SUBTITLE_FONT_CANDIDATES = (
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
)


# Bundled font (repo asset) — works offline on any machine, incl. font-less hosts
BUNDLED_FONT = BASE_DIR / "assets" / "fonts" / "NotoSans-Bold.ttf"
# Last-resort download sources (first use only, when nothing else is found)
FONT_DOWNLOAD_URLS = (
    "https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSans/NotoSans-Bold.ttf",
    "https://raw.githubusercontent.com/JetBrains/JetBrainsMono/master/fonts/ttf/JetBrainsMono-Bold.ttf",
)


def _resolve_subtitle_font() -> str:
    """Resolve a font file that actually exists in the current environment.

    Priority: SUBTITLE_FONT env override -> known system paths -> matplotlib's
    bundled DejaVuSans-Bold (always present when matplotlib is installed,
    e.g. on Colab) -> first font matplotlib can find -> bundled repo font.
    Returns "" when nothing is found (callers fall back to Pillow's default
    font); never returns a path that does not exist.
    """
    candidates = [os.getenv("SUBTITLE_FONT", "").strip(), *_SUBTITLE_FONT_CANDIDATES]
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return candidate
    try:
        import matplotlib
        bundled = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSans-Bold.ttf"
        if bundled.is_file():
            return str(bundled)
    except Exception:
        pass
    try:
        from matplotlib import font_manager
        found = font_manager.findfont("DejaVu Sans:bold", fallback_to_default=True)
        if found and Path(found).is_file():
            return found
    except Exception:
        pass
    if BUNDLED_FONT.is_file():
        return str(BUNDLED_FONT)
    return ""


SUBTITLE_FONT = _resolve_subtitle_font()
# None = not tried yet, "" = tried and failed (never retry), str = resolved path
_subtitle_font_cache: "str | None" = SUBTITLE_FONT or None


def get_subtitle_font() -> str:
    """Return a usable font path, downloading one on first use as last resort.

    The download is attempted at most once per process. Returns "" when no
    font can be obtained — the caller then renders text with Pillow's default.
    """
    global _subtitle_font_cache
    if _subtitle_font_cache is None:
        dest = TMP_DIR / "fonts" / "NotoSans-Bold.ttf"
        if not dest.is_file():
            dest.parent.mkdir(parents=True, exist_ok=True)
            for url in FONT_DOWNLOAD_URLS:
                try:
                    urllib.request.urlretrieve(url, dest)
                    if dest.is_file() and dest.stat().st_size > 100_000:
                        break
                except Exception:
                    continue
        if dest.is_file() and dest.stat().st_size > 100_000:
            _subtitle_font_cache = str(dest)
        else:
            _subtitle_font_cache = ""
    return _subtitle_font_cache


# Subtitle style - TikTok karaoke/accordion (yellow on black, word pop)
SUBTITLE_STYLE = os.getenv("SUBTITLE_STYLE", "karaoke")  # karaoke | caption
SUBTITLE_FONT_SIZE = 52
SUBTITLE_COLOR = os.getenv("SUBTITLE_COLOR", "yellow")
SUBTITLE_EMPHASIS_COLOR = "white"  # currently spoken word highlight
SUBTITLE_STROKE_COLOR = "black"
SUBTITLE_STROKE_WIDTH = 4
SUBTITLE_POSITION = "center"  # x position
SUBTITLE_Y_OFFSET = 150  # Pixels from bottom for TikTok style
SUBTITLE_BG_COLOR = (0, 0, 0, 255)  # opaque black box (TikTok style)
SUBTITLE_WORDS_PER_GROUP = int(os.getenv("SUBTITLE_WORDS_PER_GROUP", "3"))

# Scene transitions (crossfade) + audio-scene sync padding
SCENE_TRANSITION = float(os.getenv("SCENE_TRANSITION", "0.4"))  # crossfade s
SCENE_AUDIO_PADDING = 0.4  # extra seconds appended to scene duration after TTS

# Gemini settings
GEMINI_MODEL = "gemini-3.5-flash-lite"

# Scene image generation settings (Cloudflare-only by default)
IMAGE_PROVIDER = os.getenv("IMAGE_PROVIDER", "cloudflare")  # "cloudflare" | "gemini"
GEMINI_IMAGE_MODEL = os.getenv("GEMINI_IMAGE_MODEL", "gemini-2.5-flash-image")
IMAGE_CANDIDATES = int(os.getenv("IMAGE_CANDIDATES", "1"))  # candidates per scene (1 = one image each)
IMAGE_MAX_RETRIES = 3
# Interactive image approval (Enter=approve, d=change, s=skip) — off by default
APPROVE_IMAGES = os.getenv("APPROVE_IMAGES", "false").lower() == "true"
IMAGE_WIDTH = 1080
IMAGE_HEIGHT = 1920
# Fallback when Gemini quota is exhausted: keyless Pollinations.ai image API
IMAGE_FALLBACK_POLLINATIONS = os.getenv("IMAGE_FALLBACK_POLLINATIONS", "true").lower() == "true"
POLLINATIONS_TIMEOUT = int(os.getenv("POLLINATIONS_TIMEOUT", "90"))
# Fallback: Cloudflare Workers AI text-to-image (needs ACCOUNT_ID + API token;
# call is plain REST — no wrangler needed)
IMAGE_FALLBACK_CLOUDFLARE = os.getenv("IMAGE_FALLBACK_CLOUDFLARE", "true").lower() == "true"
CLOUDFLARE_ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID", "")
CLOUDFLARE_API_TOKEN = os.getenv("CLOUDFLARE_API_TOKEN", "")
CLOUDFLARE_IMAGE_MODEL = os.getenv(
    "CLOUDFLARE_IMAGE_MODEL", "@cf/bytedance/stable-diffusion-xl-lightning")
CLOUDFLARE_TIMEOUT = int(os.getenv("CLOUDFLARE_TIMEOUT", "90"))

# Google Colab CLI settings
COLAB_ENABLED = os.getenv("COLAB_ENABLED", "false").lower() == "true"
COLAB_GPU = os.getenv("COLAB_GPU", "")  # e.g., T4, L4, G4, H100, A100
COLAB_TPU = os.getenv("COLAB_TPU", "")  # e.g., v5e1, v6e1
COLAB_SESSION_NAME = os.getenv("COLAB_SESSION_NAME", "shorts-gen")
COLAB_HIGH_MEM = os.getenv("COLAB_HIGH_MEM", "false").lower() == "true"


In [ ]:
%%writefile pipeline/__init__.py


In [ ]:
%%writefile pipeline/shorts_pipeline.py
import sys
from typing import List, Optional
from pathlib import Path

from services.scene_generator import SceneGenerator, Scene
from services.image_generator import ImageGenerator
from services.voice_generator import VoiceGenerator
from services.subtitle_sync import SubtitleSync
from services.video_assembler import VideoAssembler

import config


class ShortsPipeline:
    def __init__(self):
        self.scene_generator = SceneGenerator()
        self.image_generator = ImageGenerator()
        self.voice_generator = VoiceGenerator()
        self.subtitle_sync = SubtitleSync()
        self.video_assembler = VideoAssembler()
        self.scenes: List[Scene] = []

    def run(self, topic: str, interactive: Optional[bool] = None) -> str:
        """Run the complete pipeline to generate a YouTube Shorts video.

        interactive: image approval mode for step 2. None = auto-detect
        (interactive only when APPROVE_IMAGES is on and stdin is a TTY).
        """
        if interactive is None:
            interactive = config.APPROVE_IMAGES and sys.stdin.isatty()

        print(f"\n{'='*60}")
        print(f"YouTube Shorts Generator")
        print(f"Topic: {topic}")
        print(f"{'='*60}\n")

        # Step 1: Generate scenes
        print("[1/5] Generating scenes...")
        self.scenes = self.scene_generator.generate_scenes(topic)
        self.scene_generator.validate_scenes(self.scenes)
        print(f"  ✓ Generated {len(self.scenes)} scenes")

        # Step 2: Generate scene images (with optional interactive approval)
        print("\n[2/5] Generating scene images...")
        if interactive:
            print("  Approval: [Enter]=approve · [d]=change · [s]=skip · [q]=cancel")
        self.scenes = self.image_generator.generate_all_images(
            self.scenes, interactive=interactive
        )
        print(f"  ✓ Images prepared for all scenes")

        # Step 3: Generate voiceovers
        print("\n[3/5] Generating voiceovers...")
        self.scenes = self.voice_generator.generate_all_voiceovers(self.scenes)
        print(f"  ✓ Voiceovers generated for all scenes")

        # Step 4: Sync subtitles
        print("\n[4/5] Synchronizing subtitles...")
        srt_path = self.subtitle_sync.generate_srt(self.scenes)
        print(f"  ✓ Subtitles synchronized: {srt_path}")

        # Step 5: Assemble video
        print("\n[5/5] Assembling final video...")
        output_path = self.video_assembler.assemble_video(self.scenes)
        print(f"  ✓ Video assembled: {output_path}")

        # Summary
        total_duration = sum(s.duration for s in self.scenes)
        print(f"\n{'='*60}")
        print(f"Video Generation Complete!")
        print(f"  Output: {output_path}")
        print(f"  Duration: {total_duration:.1f}s")
        print(f"  Scenes: {len(self.scenes)}")
        print(f"  Format: {config.VIDEO_WIDTH}x{config.VIDEO_HEIGHT} (9:16)")
        print(f"{'='*60}\n")

        return output_path

    def run_with_progress(self, topic: str, interactive: Optional[bool] = None) -> str:
        """Run pipeline with detailed progress updates."""
        return self.run(topic, interactive=interactive)


if __name__ == "__main__":
    import sys
    
    if len(sys.argv) < 2:
        print("Usage: python -m pipeline.shorts_pipeline <topic>")
        sys.exit(1)
    
    topic = " ".join(sys.argv[1:])
    pipeline = ShortsPipeline()
    pipeline.run(topic)


In [ ]:
%%writefile services/__init__.py


In [ ]:
%%writefile services/scene_generator.py
import json
import re
import time
from typing import List, Dict, Any
from dataclasses import dataclass, field, asdict

from google import genai
from google.genai import types

import config


@dataclass
class Scene:
    id: int
    text: str
    visual_description: str
    search_query: str
    duration: float
    image_path: str = ""
    image_candidates: List[str] = field(default_factory=list)
    audio_path: str = ""
    # Exact word timings from Edge TTS WordBoundary events (seconds,
    # audio-relative): [{"word": str, "start": float, "end": float}, ...]
    word_timings: List[Dict[str, Any]] = field(default_factory=list)


PROMPT_TEMPLATE = """Sen bir YouTube Shorts video senaryo yazarısın. "{topic}" konusunda kısa, ilgi çekici bir video senaryosu yaz.

Kriterler:
- Toplam 5-8 sahne oluşturun
- Her sahne 6-12 saniye arasında olmalı
- Toplam süre 40-60 saniye arasında olmalı
- Her sahne için kısa, net bir metin yaz (Türkçe)
- Her sahne için görsel açıklama yaz (İngilizce, sahne görselini üretmek için)
- Her sahne için Gemini görsel üretiminde kullanılacak İngilizce anahtar kelimeler belirle
- Görsel dikey format (9:16) için uygun olmalı

Cevabını SADECE şu JSON formatında ver, başka hiçbir şey ekleme:
{{
  "scenes": [
    {{
      "id": 1,
      "text": "Sahne metni (Türkçe)",
      "visual_description": "Visual description in English",
      "search_query": "English keywords for scene image generation",
      "duration": 8.0
    }}
  ]
}}
"""


class SceneGenerator:
    def __init__(self, api_key: str = None):
        self.api_key = api_key or config.GEMINI_API_KEY
        if not self.api_key:
            raise ValueError("GEMINI_API_KEY environment variable is not set. Set it in .env file.")
        self.client = genai.Client(api_key=self.api_key)
        self.model_name = config.GEMINI_MODEL

    def _extract_json(self, text: str) -> dict:
        """Extract JSON from Gemini response, handling markdown code blocks."""
        text = text.strip()
        # Try to find JSON in markdown code blocks first
        json_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', text, re.DOTALL)
        if json_match:
            text = json_match.group(1).strip()
        return json.loads(text)

    def generate_scenes(self, topic: str, max_retries: int = 3) -> List[Scene]:
        """Generate 5-8 scenes from a topic using Google Gemini API."""
        prompt = PROMPT_TEMPLATE.format(topic=topic)

        config_obj = types.GenerateContentConfig(
            system_instruction="Sen bir YouTube Shorts video senaryo uzmanısın. Sadece JSON formatında cevap ver.",
            temperature=0.7,
            max_output_tokens=2000,
            response_mime_type="application/json"
        )

        for attempt in range(max_retries):
            try:
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=config_obj
                )

                content = response.text
                data = self._extract_json(content)

                if "scenes" not in data:
                    raise ValueError("Response does not contain 'scenes' key")

                scenes = []
                for scene_data in data["scenes"]:
                    scene = Scene(
                        id=int(scene_data["id"]),
                        text=str(scene_data["text"]).strip(),
                        visual_description=str(scene_data["visual_description"]).strip(),
                        search_query=str(scene_data["search_query"]).strip(),
                        duration=float(scene_data["duration"])
                    )
                    scenes.append(scene)

                # Validate total duration
                total_duration = sum(s.duration for s in scenes)
                if total_duration < config.MIN_DURATION or total_duration > config.MAX_DURATION:
                    # Adjust durations proportionally
                    target_duration = (config.MIN_DURATION + config.MAX_DURATION) / 2  # 50 seconds
                    scale_factor = target_duration / total_duration
                    for scene in scenes:
                        scene.duration = round(scene.duration * scale_factor, 1)

                print(f"Generated {len(scenes)} scenes (total duration: {sum(s.duration for s in scenes):.1f}s)")
                return scenes

            except Exception as e:
                print(f"Attempt {attempt + 1}/{max_retries} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                else:
                    raise RuntimeError(f"Failed to generate scenes after {max_retries} attempts: {e}")

    def validate_scenes(self, scenes: List[Scene]) -> bool:
        """Validate that scenes meet the requirements."""
        if len(scenes) < config.MIN_SCENES or len(scenes) > config.MAX_SCENES:
            print(f"Warning: Scene count {len(scenes)} is outside recommended range ({config.MIN_SCENES}-{config.MAX_SCENES})")

        total_duration = sum(s.duration for s in scenes)
        if total_duration < config.MIN_DURATION:
            print(f"Warning: Total duration {total_duration:.1f}s is below minimum ({config.MIN_DURATION}s)")
        elif total_duration > config.MAX_DURATION:
            print(f"Warning: Total duration {total_duration:.1f}s exceeds maximum ({config.MAX_DURATION}s)")

        return True


if __name__ == "__main__":
    # Test the scene generator
    generator = SceneGenerator()
    scenes = generator.generate_scenes("Galaksi kaç yaşında?")
    for scene in scenes:
        print(f"\nScene {scene.id}:")
        print(f"  Text: {scene.text}")
        print(f"  Visual: {scene.visual_description}")
        print(f"  Search: {scene.search_query}")
        print(f"  Duration: {scene.duration}s")


In [ ]:
%%writefile services/image_generator.py
"""Scene image generation (Cloudflare-only by default, Gemini opt-in).

Primary provider is Cloudflare Workers AI (`IMAGE_PROVIDER=cloudflare`);
Gemini chain (`IMAGE_PROVIDER=gemini`) is kept as opt-in. Every saved
image is validated (decodes + not blank) before acceptance. Approval flow
is provider-agnostic: candidates are plain file paths either way.
"""
import base64
import hashlib
import json
import random
import subprocess
import time
import urllib.parse
import urllib.request
from datetime import datetime
from pathlib import Path
from typing import List, Optional

from google import genai
from google.genai import types as genai_types

import config
from services.scene_generator import Scene


class ImageGenerator:
    """Generates portrait scene images with Gemini and lets the user approve them.

    Approval loop (interactive mode): Enter=approve, d=change (next candidate /
    regenerate batch), s=skip scene (black background), q=cancel the whole run.
    When stdin is not a TTY (Colab/CI/remote) the first candidate is auto-approved.
    """

    def __init__(self, api_key: str = None, model: str = None):
        self.api_key = api_key or config.GEMINI_API_KEY
        self.provider = (getattr(config, "IMAGE_PROVIDER", "cloudflare") or "cloudflare").lower()
        if self.provider == "gemini" and not self.api_key:
            raise ValueError("GEMINI_API_KEY environment variable is not set. Set it in .env file.")
        self.client = genai.Client(api_key=self.api_key) if self.api_key else None
        # Comma-separated env value defines the failover order (config: "m1,m2")
        self.model_name = model or getattr(config, "GEMINI_IMAGE_MODEL", "gemini-2.5-flash-image")
        self.model_names = [m.strip() for m in str(self.model_name).split(",") if m.strip()]
        self._model_idx = 0
        self.output_dir = config.IMAGES_DIR
        self.output_dir.mkdir(parents=True, exist_ok=True)
        # Fail-fast flag: once every Gemini model quota is exhausted, stop calling the API
        self._quota_limited = False

    # ------------------------------------------------------------------ prompt
    def build_prompt(self, scene: Scene) -> str:
        """Build the Turkish image prompt for a scene (visual hints stay English)."""
        return (
            f"Bu sahneyi görselleştir: {scene.visual_description}. "
            f"Görsel anahtar kelimeler: {scene.search_query}. "
            f"YouTube Short için dikey 9:16 kompozisyon ({config.IMAGE_WIDTH}x{config.IMAGE_HEIGHT}), "
            "sinematik, yüksek detay, metin, filigran veya çerçeve içermeyen."
        )

    # -------------------------------------------------------------- generation
    @staticmethod
    def _is_quota_error(exc: Exception) -> bool:
        text = str(exc)
        return "RESOURCE_EXHAUSTED" in text or "429" in text

    @staticmethod
    def _extract_image_bytes(response) -> Optional[tuple]:
        """Return (bytes, mime_type) of the first inline image in the response."""
        try:
            parts = response.candidates[0].content.parts
        except (AttributeError, IndexError, TypeError):
            return None
        for part in parts:
            inline = getattr(part, "inline_data", None)
            if inline and inline.data:
                return inline.data, inline.mime_type or "image/png"
        return None

    def _save_image(self, scene: Scene, data: bytes, mime_type: str, index: int) -> Path:
        ext = ".jpg" if "jpeg" in mime_type else ".png"
        stamp = datetime.now().strftime("%Y%m%d%H%M%S")
        path = self.output_dir / f"scene_{scene.id:02d}_{stamp}_c{index}{ext}"
        path.write_bytes(data)
        return path

    @staticmethod
    def _is_blank_image(data: bytes) -> bool:
        """True when the image decodes but is blank (solid color / tiny variance).

        Downscales to 32x32 grayscale and checks pixel stddev; a real
        diffusion photo always has texture, a failed solid fill has ~0.
        Undecodable data is also treated as blank (rejected).
        """
        try:
            from PIL import Image
            import io
            import statistics
            with Image.open(io.BytesIO(data)) as im:
                small = im.convert("L").resize((32, 32))
                px = list(small.getdata())
            return statistics.pstdev(px) < 3.0
        except Exception:
            return True

    def _accept_image(self, scene: Scene, data: bytes, mime_type: str,
                      index: int, tag: str) -> Optional[str]:
        """Validate + save one image; returns path or None when rejected."""
        if self._is_blank_image(data):
            print(f"  {tag}: boş resim reddedildi (yeniden deneniyor)")
            return None
        saved = self._save_image(scene, data, mime_type, index=index)
        print(f"  {tag} image: {saved.name}")
        return str(saved)

    def generate_candidates(self, scene: Scene, count: int = None,
                            max_retries: int = None) -> List[str]:
        """Generate up to `count` candidate images for a scene.

        Cloudflare-only by default (`IMAGE_PROVIDER=cloudflare`): validated
        Cloudflare images, then gradient when unavailable. With
        `IMAGE_PROVIDER=gemini` the old chain runs: Gemini model list, then
        Cloudflare + Pollinations fallbacks on quota (429 RESOURCE_EXHAUSTED).
        Returns the list of downloaded file paths (may be empty).
        """
        count = count or config.IMAGE_CANDIDATES
        max_retries = max_retries or config.IMAGE_MAX_RETRIES
        if self.provider != "gemini":
            if not self._cloudflare_available():
                print("  ✗ Cloudflare kullanılamıyor → .env'e ekleyin:\n"
                      "      CLOUDFLARE_ACCOUNT_ID=...\n"
                      "      CLOUDFLARE_API_TOKEN=...\n"
                      "    (config IMAGE_PROVIDER=cloudflare, flag "
                      f"{config.IMAGE_FALLBACK_CLOUDFLARE}, id var="
                      f"{bool(config.CLOUDFLARE_ACCOUNT_ID)}, token var="
                      f"{bool(config.CLOUDFLARE_API_TOKEN)}) → gradient")
                return []
            return self._generate_cloudflare(scene, count)
        paths: List[str] = []
        if not self._quota_limited:
            paths = self._generate_gemini(scene, count, max_retries)
        if self._quota_limited and len(paths) < count:
            if self._cloudflare_available():
                need = count - len(paths)
                print(f"  Gemini kotası dolu → Cloudflare Workers AI fallback ({need} aday)...")
                paths += self._generate_cloudflare(scene, need)
            if len(paths) < count and config.IMAGE_FALLBACK_POLLINATIONS:
                need = count - len(paths)
                print(f"  Gemini kotası dolu → Pollinations.ai fallback ({need} aday)...")
                paths += self._generate_pollinations(scene, need)
        return paths

    def _generate_gemini(self, scene: Scene, count: int, max_retries: int) -> List[str]:
        """Gemini generation with in-list model failover on quota errors."""
        paths: List[str] = []
        attempts = 0
        max_attempts = max_retries * count
        while (len(paths) < count and attempts < max_attempts
               and not self._quota_limited):
            attempts += 1
            try:
                response = self.client.models.generate_content(
                    model=self.model_names[self._model_idx],
                    contents=self.build_prompt(scene),
                    config=genai_types.GenerateContentConfig(
                        response_modalities=["IMAGE", "TEXT"],
                    ),
                )
            except Exception as exc:  # noqa: BLE001 - API errors vary by type
                if self._is_quota_error(exc):
                    self._model_idx += 1
                    if self._model_idx >= len(self.model_names):
                        self._quota_limited = True
                        print("  ✗ Gemini görsel kotası doldu (429 RESOURCE_EXHAUSTED) — "
                              "tüm modeller tükendi.")
                        return paths
                    nxt = self.model_names[self._model_idx]
                    print(f"  Kota doldu → sonraki model deneniyor: {nxt}")
                    attempts -= 1  # quota failover must not consume the budget
                    continue
                print(f"  Image generation error (attempt {attempts}): {exc}")
                time.sleep(2)
                continue

            image = self._extract_image_bytes(response)
            if image is None:
                print(f"  No image in model response (attempt {attempts})")
                continue

            data, mime_type = image
            accepted = self._accept_image(scene, data, mime_type,
                                          index=len(paths) + 1,
                                          tag="Generated")
            if accepted is None:
                continue
            paths.append(accepted)

        return paths

    # ------------------------------------------------------- pollinations fb
    def _build_pollinations_prompt(self, scene: Scene) -> str:
        """English-oriented prompt (Pollinations models prefer English)."""
        return (
            f"{scene.search_query}. Cinematic vertical 9:16 composition, "
            "high detail, no text, no watermark"
        )

    def _generate_pollinations(self, scene: Scene, count: int) -> List[str]:
        """Keyless fallback: pollinations.ai GET image API (no account)."""
        paths: List[str] = []
        prompt = urllib.parse.quote(self._build_pollinations_prompt(scene))
        for _ in range(count):
            seed = random.randint(1, 2 ** 31)
            url = (
                f"https://image.pollinations.ai/prompt/{prompt}"
                f"?width={config.IMAGE_WIDTH}&height={config.IMAGE_HEIGHT}"
                f"&nologo=true&seed={seed}"
            )
            for attempt in range(2):
                try:
                    req = urllib.request.Request(
                        url, headers={"User-Agent": "appShorts/1.0"})
                    with urllib.request.urlopen(
                            req, timeout=config.POLLINATIONS_TIMEOUT) as resp:
                        data = resp.read()
                    if len(data) < 5_000:
                        raise RuntimeError(f"tiny response ({len(data)} bytes)")
                    accepted = self._accept_image(scene, data, "image/jpeg",
                                                  index=len(paths) + 1,
                                                  tag="Pollinations")
                    if accepted is None:
                        break  # blank rejected: don't retry same seed twice
                    paths.append(accepted)
                    break
                except Exception as exc:  # noqa: BLE001 - urllib errors vary
                    if attempt == 0:
                        time.sleep(3)
                        continue
                    print(f"  Pollinations error: {exc}")
        return paths

    # -------------------------------------------------------- cloudflare fb
    @staticmethod
    def _cloudflare_available() -> bool:
        """Cloudflare fallback needs flag + ACCOUNT_ID + API token."""
        return (bool(config.IMAGE_FALLBACK_CLOUDFLARE)
                and bool(config.CLOUDFLARE_ACCOUNT_ID)
                and bool(config.CLOUDFLARE_API_TOKEN))

    def _generate_cloudflare(self, scene: Scene, count: int) -> List[str]:
        """Primary provider: Cloudflare Workers AI text-to-image REST API.

        POSTs to .../accounts/{id}/ai/run/{model} with a Bearer token.
        The response is a binary image (usual) or JSON holding base64.
        Blank images are rejected and retried once with a fresh prompt seed.
        Returns saved paths (may be empty on error); never raises.
        """
        paths: List[str] = []
        if not self._cloudflare_available():
            return paths
        base_prompt = self._build_pollinations_prompt(scene)
        url = (f"https://api.cloudflare.com/client/v4/accounts/"
               f"{config.CLOUDFLARE_ACCOUNT_ID}/ai/run/"
               f"{config.CLOUDFLARE_IMAGE_MODEL}")
        for _ in range(count):
            data, mime = None, "image/png"
            for retry in range(2):  # blank rejected -> 1 retry, new seed
                prompt = base_prompt if retry == 0 else f"{base_prompt}, seed variation {random.randint(1, 999999)}"
                body = json.dumps({
                    "prompt": prompt,
                    "num_steps": 6,
                }).encode("utf-8")
                try:
                    req = urllib.request.Request(
                        url, data=body,
                        headers={
                            "Authorization": f"Bearer {config.CLOUDFLARE_API_TOKEN}",
                            "Content-Type": "application/json",
                            "User-Agent": "appShorts/1.0",
                        })
                    with urllib.request.urlopen(
                            req, timeout=config.CLOUDFLARE_TIMEOUT) as resp:
                        ctype = resp.headers.get("Content-Type", "")
                        raw = resp.read()
                    if "json" in ctype:
                        payload = json.loads(raw.decode("utf-8", "replace"))
                        b64 = (payload.get("result") or {}).get("image", "")
                        if not b64:
                            raise RuntimeError("no image in JSON response")
                        data = base64.b64decode(b64)
                        mime = "image/png"
                    else:
                        data = raw
                        mime = "image/png" if "png" in ctype else "image/jpeg"
                    if self._is_blank_image(data):
                        print("  Cloudflare: boş resim reddedildi (yeniden deneniyor)")
                        data = None
                        continue
                    if len(data) < 5_000:
                        raise RuntimeError(f"tiny response ({len(data)} bytes)")
                    break
                except Exception as exc:  # noqa: BLE001 - urllib errors vary
                    print(f"  Cloudflare error: {exc}")
                    data = None
                    break
            if not data:
                continue
            saved = self._save_image(scene, data, mime,
                                     index=len(paths) + 1)
            paths.append(str(saved))
            print(f"  Cloudflare image: {saved.name}")
        return paths

    # ----------------------------------------------------- gradient (offline)
    def _generate_gradient(self, scene: Scene) -> str:
        """Offline last resort: deterministic two-color vertical gradient.

        Replaces the plain black background when every provider failed, so
        the video still has a colored, topic-seeded backdrop.
        """
        import colorsys
        from PIL import Image

        digest = hashlib.md5(scene.search_query.encode("utf-8")).digest()
        top = colorsys.hsv_to_rgb(digest[0] / 255.0, 0.55, 0.35)
        bottom = colorsys.hsv_to_rgb(digest[1] / 255.0, 0.75, 0.85)
        h, w = config.IMAGE_HEIGHT, config.IMAGE_WIDTH
        strip = Image.new("RGB", (1, h))
        for y in range(h):
            t = y / max(h - 1, 1)
            strip.putpixel((0, y), tuple(
                int(a + (b - a) * t) for a, b in zip(top, bottom)))
        img = strip.resize((w, h))
        stamp = datetime.now().strftime("%Y%m%d%H%M%S")
        path = self.output_dir / f"scene_{scene.id:02d}_{stamp}_gradient.png"
        img.save(path)
        print(f"  Procedural gradient: {path.name}")
        return str(path)

    # ---------------------------------------------------------------- approval
    @staticmethod
    def _open_in_viewer(path: str) -> bool:
        """Try to open the image in the platform's default viewer."""
        for cmd in (["termux-open", str(path)], ["xdg-open", str(path)], ["open", str(path)]):
            try:
                subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                return True
            except (FileNotFoundError, OSError):
                continue
        return False

    def approve_candidates(self, scene: Scene, candidates: List[str],
                           interactive: bool = True) -> Optional[str]:
        """Interactive approval: returns approved path, None = skip (black bg).

        Raises KeyboardInterrupt when the user cancels with 'q'.
        """
        if not candidates:
            return None
        if not interactive:
            print(f"  Selected image: {Path(candidates[0]).name} (auto-approved)")
            return candidates[0]

        idx = 0
        extra_rounds = 0
        while True:
            path = candidates[idx]
            print(f"\n  Scene {scene.id} — candidate {idx + 1}/{len(candidates)}: {path}")
            self._open_in_viewer(path)
            try:
                choice = input(
                    "    [Enter] Onayla  ·  [d] Değiştir  ·  [s] Atla (siyah zemin)  ·  [q] İptal > "
                ).strip().lower()
            except EOFError:
                # Non-interactive stdin (piped input): accept what is on screen
                return path

            if choice in ("", "o", "onayla"):
                return path
            if choice in ("q", "iptal"):
                raise KeyboardInterrupt("Kullanıcı tarafından iptal edildi.")
            if choice in ("s", "atla"):
                print("  Scene skipped → black background.")
                return None
            if choice in ("d", "degistir", "değiştir"):
                idx += 1
                if idx >= len(candidates):
                    if extra_rounds < 2 and not self._quota_limited:
                        extra_rounds += 1
                        print("  All candidates shown — generating a new batch...")
                        fresh = self.generate_candidates(scene)
                        if fresh:
                            candidates = fresh
                            scene.image_candidates = fresh
                            idx = 0
                            continue
                    idx = 0  # cycle back through existing candidates
                continue
            print("  Unrecognized key — Enter/d/s/q use.")

    # ------------------------------------------------------------------- main
    def generate_all_images(self, scenes: List[Scene], interactive: bool = False) -> List[Scene]:
        """Generate images for all scenes with optional per-scene approval."""
        total = len(scenes)
        for i, scene in enumerate(scenes):
            print(f"\n[{i + 1}/{total}] Generating images for scene {scene.id}: {scene.search_query}")
            candidates = self.generate_candidates(scene)
            scene.image_candidates = candidates

            if not candidates:
                # Last resort: colored procedural background instead of black
                gradient = self._generate_gradient(scene)
                scene.image_path = gradient
                if self._quota_limited:
                    print(f"  Scene {scene.id}: quota exhausted → gradient background")
                else:
                    print(f"  Warning: No image generated for scene {scene.id} "
                          f"→ gradient background")
                continue

            selected = self.approve_candidates(scene, candidates, interactive=interactive)
            scene.image_path = selected or ""
            if selected:
                print(f"  ✓ Scene {scene.id}: {Path(selected).name}")
            else:
                print(f"  Scene {scene.id}: no approved image → black background")

        return scenes


if __name__ == "__main__":
    # Manual smoke test: python -m services.image_generator
    gen = ImageGenerator()
    demo = Scene(id=1, text="test", visual_description="cinematic medieval castle",
                 search_query="medieval castle", duration=6.0)
    print("Prompt:", gen.build_prompt(demo))
    print("Candidates:", gen.generate_candidates(demo, count=1))



In [ ]:
%%writefile services/voice_generator.py
import asyncio
import time
import edge_tts
from typing import Dict, List, Optional
from pathlib import Path

import config
from services.scene_generator import Scene


def _run_coro(coro):
    """Run a coroutine from sync code, even inside a running event loop.

    Jupyter/Colab (ipykernel) keeps a live event loop, where asyncio.run()
    raises RuntimeError. In that case the coroutine is executed on a fresh
    loop in a worker thread instead. Outside a loop (plain CLI) the standard
    asyncio.run() path is used unchanged.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


class VoiceGenerator:
    def __init__(self, voice_name: str = None):
        self.voice_name = voice_name or config.VOICE_NAME
        self.output_dir = config.AUDIO_DIR
        self.output_dir.mkdir(parents=True, exist_ok=True)

    async def _generate_single(self, text: str, output_path: str, rate: str = "+0%") -> Optional[str]:
        """Generate a single voiceover using Edge TTS.

        Streams the synthesis so WordBoundary events (exact per-word offsets)
        can be captured; they are written next to the audio as
        ``<name>.words.json`` and returned to the caller for subtitle sync.
        """
        import json as _json
        words: List[Dict] = []
        try:
            communicate = edge_tts.Communicate(
                text, self.voice_name, rate=rate, boundary="WordBoundary")
            audio = bytearray()
            async for chunk in communicate.stream():
                if chunk["type"] == "audio":
                    audio.extend(chunk["data"])
                elif chunk["type"] == "WordBoundary":
                    # offset/duration are in 100-nanosecond ticks
                    start = chunk["offset"] / 1e7
                    end = start + chunk["duration"] / 1e7
                    words.append({
                        "word": chunk["text"],
                        "start": round(start, 3),
                        "end": round(end, 3),
                    })

            if not audio:
                print(f"  Warning: Edge TTS returned no audio: {output_path}")
                return None
            Path(output_path).write_bytes(bytes(audio))
            if Path(output_path).exists() and Path(output_path).stat().st_size > 0:
                if words:
                    words_path = str(Path(output_path).with_suffix(".words.json"))
                    try:
                        Path(words_path).write_text(
                            _json.dumps(words, ensure_ascii=False), encoding="utf-8")
                    except Exception as e:
                        print(f"  Warning: Could not save word timings: {e}")
                return output_path
            else:
                print(f"  Warning: Generated audio file is empty: {output_path}")
                return None
        except Exception as e:
            print(f"  Error generating voiceover: {e}")
            return None

    def generate_voiceover(self, scene: Scene, max_retries: int = None) -> Optional[str]:
        """Generate voiceover for a single scene.

        Edge TTS intermittently returns HTTP 403 (rate-limit/throttle) on the
        WSS connect when scenes are synthesized back-to-back. Retries use
        exponential backoff so the throttle window can pass.
        """
        if max_retries is None:
            try:
                max_retries = int(getattr(config, "TTS_MAX_RETRIES", 5))
            except (TypeError, ValueError):
                max_retries = 5
        try:
            base_delay = float(getattr(config, "TTS_RETRY_BASE_DELAY", 3.0))
        except (TypeError, ValueError):
            base_delay = 3.0
        filename = f"scene_{scene.id:02d}.mp3"
        output_path = str(self.output_dir / filename)

        # Log retries only when the first attempt already fails — this
        # surfaces the real error instead of 5 identical "attempt X failed"
        # lines, and makes "I couldn't generate sound in 5 tries" diagnosable.
        first_failed = None
        for attempt in range(max_retries):
            try:
                result = _run_coro(self._generate_single(scene.text, output_path))
                if result:
                    print(f"  Generated voiceover: {filename}")
                    return result
                print(f"  Attempt {attempt + 1}/{max_retries} returned no audio, retrying...")
            except Exception as e:
                if attempt == 0:
                    first_failed = e  # save the real error for final message
                print(f"  Attempt {attempt + 1}/{max_retries} failed: {e}")

            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  Waiting {delay:.0f}s before retry...")
                time.sleep(delay)

        if first_failed is not None:
            print(f"  ✗ Could not generate voiceover for scene {scene.id} "
                  f"after {max_retries} attempts: {first_failed}")
            print("  Hints: check your GEMINI_API_KEY (TTS needs no key) and "
                  "network access to services.google.com; try again.")
        return None

    @staticmethod
    def _load_word_timings(audio_path: str) -> List[Dict]:
        """Load Edge TTS WordBoundary timings saved next to the audio file."""
        import json as _json
        words_path = str(Path(audio_path).with_suffix(".words.json"))
        try:
            if Path(words_path).is_file():
                data = _json.loads(Path(words_path).read_text(encoding="utf-8"))
                if isinstance(data, list):
                    return data
        except Exception as e:
            print(f"  Warning: Could not load word timings: {e}")
        return []

    def generate_all_voiceovers(self, scenes: List[Scene]) -> List[Scene]:
        """Generate voiceovers for all scenes and sync scene durations to audio.

        Edge TTS output length rarely matches the LLM-estimated scene.duration;
        using the estimate cuts speech off at scene transitions. After each
        successful TTS the scene duration becomes the real audio length plus
        a small padding (SCENE_AUDIO_PADDING) so transitions never interrupt.
        """
        for i, scene in enumerate(scenes):
            print(f"\n[{i + 1}/{len(scenes)}] Generating voiceover for scene {scene.id}")
            if i > 0:
                try:
                    scene_delay = float(getattr(config, "TTS_SCENE_DELAY", 1.5))
                except (TypeError, ValueError):
                    scene_delay = 1.5
                time.sleep(scene_delay)
            audio_path = self.generate_voiceover(scene)
            if audio_path:
                scene.audio_path = audio_path
                scene.word_timings = self._load_word_timings(audio_path)
                if scene.word_timings:
                    print(f"  Word timings: {len(scene.word_timings)} words")
                actual = self.get_audio_duration(audio_path)
                if actual and actual > 0:
                    scene.duration = round(actual + getattr(config, "SCENE_AUDIO_PADDING", 0.4), 2)
                    print(f"  Scene {scene.id} duration synced to audio: {scene.duration:.2f}s")
            else:
                print(f"  Warning: Could not generate voiceover for scene {scene.id}")

        return scenes

    def get_audio_duration(self, audio_path: str) -> float:
        """Get the duration of an audio file in seconds."""
        try:
            from moviepy import AudioFileClip
            with AudioFileClip(audio_path) as clip:
                return clip.duration
        except Exception as e:
            print(f"  Error getting audio duration: {e}")
            return 0.0


if __name__ == "__main__":
    # Test the voice generator
    generator = VoiceGenerator()
    
    test_scene = Scene(
        id=1,
        text="Yapay zeka, makinelerin insan benzeri zekâ göstermesini sağlayan bir teknolojidir.",
        visual_description="AI technology visualization",
        search_query="artificial intelligence",
        duration=8.0
    )
    
    audio_path = generator.generate_voiceover(test_scene)
    if audio_path:
        duration = generator.get_audio_duration(audio_path)
        print(f"Audio duration: {duration:.2f}s")


In [ ]:
%%writefile services/subtitle_sync.py
import re
from typing import List, Tuple
from pathlib import Path

import config
from services.scene_generator import Scene
from services.voice_generator import VoiceGenerator


class SubtitleSync:
    def __init__(self):
        self.voice_generator = VoiceGenerator()
        self.max_chars_per_line = 40  # Max characters per subtitle line

    def split_text(self, text: str, max_chars: int = None) -> List[str]:
        """Split long text into multiple lines for subtitles."""
        max_chars = max_chars or self.max_chars_per_line
        
        if len(text) <= max_chars:
            return [text]
        
        lines = []
        words = text.split()
        current_line = ""
        
        for word in words:
            if len(current_line) + len(word) + 1 <= max_chars:
                if current_line:
                    current_line += " " + word
                else:
                    current_line = word
            else:
                lines.append(current_line)
                current_line = word
        
        if current_line:
            lines.append(current_line)
        
        return lines

    def format_time(self, seconds: float) -> str:
        """Format seconds to SRT time format (HH:MM:SS,mmm)."""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds % 1) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

    def calculate_subtitle_timing(self, scenes: List[Scene]) -> List[dict]:
        """Calculate subtitle timing for all scenes based on audio duration."""
        subtitles = []
        current_time = 0.0
        
        for scene in scenes:
            if not scene.audio_path or not Path(scene.audio_path).exists():
                # Use scene duration if no audio
                duration = scene.duration
            else:
                # Get actual audio duration
                duration = self.voice_generator.get_audio_duration(scene.audio_path)
                if duration <= 0:
                    duration = scene.duration
            
            # Split text into lines if needed
            lines = self.split_text(scene.text)
            
            # Calculate time per line
            time_per_line = duration / max(len(lines), 1)
            
            for i, line in enumerate(lines):
                start = current_time + (i * time_per_line)
                end = start + time_per_line
                
                subtitles.append({
                    "index": len(subtitles) + 1,
                    "start": start,
                    "end": end,
                    "text": line,
                    "scene_id": scene.id
                })
            
            current_time += duration
        
        return subtitles

    def generate_srt(self, scenes: List[Scene], output_path: str = None) -> str:
        """Generate SRT subtitle file for all scenes."""
        subtitles = self.calculate_subtitle_timing(scenes)
        
        if not output_path:
            output_path = str(config.TMP_DIR / "subtitles.srt")
        
        srt_content = ""
        for sub in subtitles:
            srt_content += f"{sub['index']}\n"
            srt_content += f"{self.format_time(sub['start'])} --> {self.format_time(sub['end'])}\n"
            srt_content += f"{sub['text']}\n\n"
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(srt_content)
        
        print(f"Generated SRT file: {output_path}")
        return output_path

    def get_subtitle_entries(self, scenes: List[Scene]) -> List[Tuple[float, float, str]]:
        """Get subtitle entries as (start, end, text) tuples for MoviePy."""
        subtitles = self.calculate_subtitle_timing(scenes)
        return [(sub["start"], sub["end"], sub["text"]) for sub in subtitles]


if __name__ == "__main__":
    # Test the subtitle sync
    sync = SubtitleSync()
    
    test_scenes = [
        Scene(
            id=1,
            text="Yapay zeka, makinelerin insan benzeri zekâ göstermesini sağlayan bir teknolojidir.",
            visual_description="AI technology",
            search_query="artificial intelligence",
            duration=8.0,
            audio_path=""
        ),
        Scene(
            id=2,
            text="Bu teknoloji, bilgisayarların öğrenme ve karar verme yeteneklerini geliştirir.",
            visual_description="Machine learning",
            search_query="machine learning",
            duration=7.0,
            audio_path=""
        )
    ]
    
    srt_path = sync.generate_srt(test_scenes)
    with open(srt_path, "r", encoding="utf-8") as f:
        print(f.read())


In [ ]:
%%writefile services/video_assembler.py
import os
from typing import List, Optional
from pathlib import Path
from datetime import timedelta

import numpy as np

from moviepy import (
    AudioFileClip,
    TextClip,
    CompositeVideoClip,
    CompositeAudioClip,
    concatenate_videoclips,
    AudioClip,
    ImageClip,
    ColorClip,
)
from moviepy.video.fx.CrossFadeIn import CrossFadeIn
from moviepy.video.fx.CrossFadeOut import CrossFadeOut

import config
from services.scene_generator import Scene
from services.subtitle_sync import SubtitleSync


class VideoAssembler:
    def __init__(self):
        self.subtitle_sync = SubtitleSync()
        self.output_dir = config.OUTPUT_DIR
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _create_subtitle_clip(self, text: str, start_time: float, end_time: float,
                                duration: float) -> TextClip:
        """Create a TikTok-style subtitle clip with scroll-up animation."""
        # Split text into lines if too long
        lines = self.subtitle_sync.split_text(text)

        # Calculate font size based on number of lines
        font_size = config.SUBTITLE_FONT_SIZE
        if len(lines) > 1:
            font_size = int(font_size * 0.8)

        # Resolve a real font; never hand Pillow a path that does not exist.
        font = config.SUBTITLE_FONT or config.get_subtitle_font() or None
        text_content = "\n".join(lines)

        def _make_text(font_path) -> TextClip:
            return TextClip(
                text=text_content,
                font=font_path,
                font_size=font_size,
                color=config.SUBTITLE_COLOR,
                stroke_color=config.SUBTITLE_STROKE_COLOR,
                stroke_width=config.SUBTITLE_STROKE_WIDTH,
                size=(config.VIDEO_WIDTH - 100, None),
                method="caption",
                text_align="center"
            )

        try:
            # Create the text clip to measure its size. If the resolved font
            # cannot be loaded (missing/corrupt), retry with Pillow's default
            # font so subtitles are never dropped.
            try:
                text_clip = _make_text(font)
            except Exception:
                text_clip = _make_text(None)

            # Get text clip dimensions
            text_w, text_h = text_clip.size

            # Semi-transparent background (MoviePy 2.x: ImageClip has no `color`
            # kwarg — build an RGBA frame whose alpha channel becomes the mask)
            bg_padding = 20
            bg_w = text_w + bg_padding * 2
            bg_h = text_h + bg_padding * 2
            bg_color = config.SUBTITLE_BG_COLOR
            bg_arr = np.zeros((bg_h, bg_w, 4), dtype=np.uint8)
            bg_arr[..., 0] = bg_color[0]
            bg_arr[..., 1] = bg_color[1]
            bg_arr[..., 2] = bg_color[2]
            bg_arr[..., 3] = bg_color[3] if len(bg_color) > 3 else 255
            bg_clip = (
                ImageClip(bg_arr)
                .with_duration(duration)
                .with_start(start_time)
            )

            # TikTok caption: static position (no scroll-up animation — a moving
            # box shimmers on every frame and re-triggers at each caption).
            # NOTE: it must never sit FULLY off-canvas — moviepy 2.x compose_mask
            # crashes on zero-overlap regions ("operands could not be broadcast").
            y_target = int(config.VIDEO_HEIGHT - config.SUBTITLE_Y_OFFSET - text_h)
            y_target = max(0, y_target)

            # Static position (integer y: no sub-pixel resampling shimmer)
            text_clip = text_clip.with_position(("center", y_target))

            # Background follows the text (static too)
            bg_clip = bg_clip.with_position(("center", y_target - bg_padding))

            # Set duration and start time
            text_clip = text_clip.with_duration(duration).with_start(start_time)

            # Return composite of bg + text
            from moviepy import CompositeVideoClip as _CVC
            # We return the text clip and handle bg separately
            # Store bg as a special attribute
            text_clip._bg_clip = bg_clip

            return text_clip

        except Exception as e:
            print(f"  Warning: Could not create subtitle clip: {e}")
            # Return empty clip as fallback
            return TextClip(
                text="",
                font=None,
                font_size=1,
                color="white",
                size=(1, 1),
                method="caption"
            ).with_duration(0.001).with_start(start_time)

    # ------------------------------------------------- karaoke (TikTok) subs
    @staticmethod
    def _measure_text_width(text: str, font_path: str, font_size: int) -> int:
        """Measure rendered text width in pixels with the actual TTF font."""
        from PIL import ImageFont
        try:
            font = ImageFont.truetype(font_path, font_size)
            return int(font.getlength(text))
        except Exception:
            return int(len(text) * font_size * 0.55)

    def _create_karaoke_word_clip(self, group_words: List[str], active_idx: int,
                                   start: float, duration: float, font: str):
        """One karaoke step: whole word group in yellow + active word in white.

        Single-canvas PIL render: box, base text and highlight are drawn onto
        ONE RGBA image, so the white word can never drift out of alignment.
        Static position: NO per-step rise/pop animation — re-triggering it on
        every word made the whole box jump (~14px) at each word change, which
        read as jitter. The box now sits still; only the white highlight
        moves between words.
        """
        from PIL import Image, ImageDraw, ImageFont

        font_size = config.SUBTITLE_FONT_SIZE
        words = list(group_words)

        try:
            pil_font = ImageFont.truetype(font, font_size) if font else ImageFont.load_default()
        except Exception:
            pil_font = ImageFont.load_default()

        stroke_w = config.SUBTITLE_STROKE_WIDTH
        tmp = Image.new("RGBA", (8, 8))
        tmp_draw = ImageDraw.Draw(tmp)

        def _w(t: str) -> float:
            try:
                return tmp_draw.textlength(t, font=pil_font, stroke_width=stroke_w)
            except TypeError:
                return tmp_draw.textlength(t, font=pil_font)

        try:
            asc, desc = pil_font.getmetrics()
            text_h = asc + desc
        except Exception:
            text_h = int(font_size * 1.25)
        try:
            space_w = _w(" ")
        except Exception:
            space_w = font_size * 0.3

        word_widths = [_w(wd) for wd in words]
        full_w = int(sum(word_widths) + space_w * max(len(words) - 1, 0))

        pad_x, pad_y = 34, 34
        box_w = int(full_w + pad_x * 2)
        box_h = max(int(text_h + pad_y * 2), int(font_size * 2.0))
        y_text = (box_h - text_h) // 2

        word_x = []
        x = float(pad_x)
        for i, wd in enumerate(words):
            word_x.append(x)
            x += word_widths[i] + (space_w if i < len(words) - 1 else 0)

        bg_color = config.SUBTITLE_BG_COLOR
        img = Image.new("RGBA", (box_w, box_h),
                        (bg_color[0], bg_color[1], bg_color[2],
                         bg_color[3] if len(bg_color) > 3 else 255))
        draw = ImageDraw.Draw(img)

        def _draw(t: str, tx: float, fill: str):
            try:
                draw.text((tx, y_text), t, font=pil_font, fill=fill,
                          stroke_width=stroke_w,
                          stroke_fill=config.SUBTITLE_STROKE_COLOR)
            except TypeError:
                draw.text((tx, y_text), t, font=pil_font, fill=fill)

        for i, wd in enumerate(words):
            color = (config.SUBTITLE_EMPHASIS_COLOR
                     if i == active_idx else config.SUBTITLE_COLOR)
            _draw(wd, word_x[i], color)

        arr = np.array(img)
        static = ImageClip(arr).with_duration(duration)

        # Integer position: float y (e.g. 1697.3) forces sub-pixel resampling
        # on every frame, which reads as shimmer. Truncate once here.
        y_final = int(config.VIDEO_HEIGHT - config.SUBTITLE_Y_OFFSET - box_h)
        y_final = max(0, y_final)

        word_clip = static.with_position(("center", y_final)).with_start(start)
        return word_clip

    def _create_karaoke_subtitles(self, scene: Scene, start_time: float) -> List:
        """TikTok accordion karaoke subtitles for one scene.

        Prefers exact Edge TTS WordBoundary timings (scene.word_timings) so
        subtitle words match the spoken audio 1:1; falls back to proportional
        estimation inside audio-synced lines when boundaries are unavailable.
        Words are grouped into fixed disjoint groups of SUBTITLE_WORDS_PER_GROUP
        (no sliding window) so no word repeats across steps.
        """
        clips = []
        font = config.SUBTITLE_FONT or config.get_subtitle_font() or None
        group_size = max(1, config.SUBTITLE_WORDS_PER_GROUP)

        # (word, start, end) tuples relative to scene audio start
        if scene.word_timings:
            word_spans = [
                (str(w.get("word", "")).strip(), float(w.get("start", 0.0)),
                 float(w.get("end", 0.0)))
                for w in scene.word_timings
                if str(w.get("word", "")).strip()
            ]
        else:
            word_spans = []
            for line_start, line_end, text in self.subtitle_sync.get_subtitle_entries([scene]):
                words = [w for w in text.split() if w]
                if not words:
                    continue
                line_dur = max(line_end - line_start, 0.001)
                weights = [max(len(w), 1) for w in words]
                total = float(sum(weights))
                t = line_start
                for i, word in enumerate(words):
                    w_dur = line_dur * weights[i] / total
                    word_spans.append((word, t, t + w_dur))
                    t += w_dur

        # Fixed disjoint groups: [0..2], [3..5], ... — zero repetition
        for gi in range(0, len(word_spans), group_size):
            group = word_spans[gi:gi + group_size]
            group_words = [g[0] for g in group]
            for j, (word, ws, we) in enumerate(group):
                w_dur = max(we - ws, 0.12)
                try:
                    clip = self._create_karaoke_word_clip(
                        group_words, active_idx=j,
                        start=start_time + ws,
                        duration=w_dur + 0.04,  # tiny overlap: no flicker
                        font=font,
                    )
                    clips.append(clip)
                except Exception as e:
                    print(f"  Error creating karaoke word '{word}': {e}")
        return clips

    def _create_image_background(self, scene: Scene) -> Optional[ImageClip]:
        """Cover-fit the generated scene image to the frame with a slow zoom.

        The image is resized to cover 1080x1920, center-cropped, then slowly
        zooms in (~6% over the scene duration) for a subtle Ken Burns effect.
        """
        img = ImageClip(scene.image_path)
        w, h = img.size
        if w <= 0 or h <= 0:
            raise ValueError(f"Invalid image size for scene {scene.id}: {img.size}")

        # Cover scale: fill the whole frame, then center-crop the overflow
        scale = max(config.VIDEO_WIDTH / w, config.VIDEO_HEIGHT / h)
        cover = img.resized(new_size=(int(round(w * scale)), int(round(h * scale))))
        cover = cover.cropped(
            x_center=cover.size[0] / 2,
            y_center=cover.size[1] / 2,
            width=config.VIDEO_WIDTH,
            height=config.VIDEO_HEIGHT,
        )

        # Slow zoom-in; oversized frames are clipped by the composite canvas
        duration = max(scene.duration, 0.001)

        def zoom_factor(t):
            return 1 + 0.06 * (t / duration)

        zoomed = cover.resized(zoom_factor)
        return (
            zoomed.with_duration(scene.duration)
            .with_position(("center", "center"))
        )

    def _create_scene_clip(self, scene: Scene, start_time: float) -> Optional[CompositeVideoClip]:
        """Create a composite clip for a single scene with image, audio, and subtitles."""
        clips = []
        
        # Background image (Gemini-generated scene image)
        bg_ok = False
        if scene.image_path and Path(scene.image_path).exists():
            try:
                bg_clip = self._create_image_background(scene).with_start(start_time)
                clips.append(bg_clip)
                bg_ok = True
            except Exception as e:
                print(f"  Error loading image for scene {scene.id}: {e}")
        if not bg_ok:
            # Black background fallback
            clips.append(
                ColorClip(
                    size=(config.VIDEO_WIDTH, config.VIDEO_HEIGHT),
                    color=(0, 0, 0),
                    duration=scene.duration
                ).with_start(start_time)
            )
        
        # Audio clip
        if scene.audio_path and Path(scene.audio_path).exists():
            try:
                audio_clip = AudioFileClip(scene.audio_path)
                
                # Trim to match scene duration
                if audio_clip.duration > scene.duration:
                    audio_clip = audio_clip.subclipped(0, scene.duration)
                
                audio_clip = audio_clip.with_start(start_time)
                
                # Add audio to video clip
                if clips:
                    clips[0] = clips[0].with_audio(audio_clip)
            except Exception as e:
                print(f"  Error loading audio for scene {scene.id}: {e}")
        
        # Subtitle clips — TikTok karaoke (accordion) or classic caption
        if config.SUBTITLE_STYLE == "karaoke":
            clips.extend(self._create_karaoke_subtitles(scene, start_time))
        else:
            subtitle_entries = self.subtitle_sync.get_subtitle_entries([scene])
            for start, end, text in subtitle_entries:
                try:
                    sub_clip = self._create_subtitle_clip(text, start, end, end - start)
                    # Semi-transparent background must come BEFORE the text clip,
                    # otherwise it composites on top and dims the text (~50%)
                    if hasattr(sub_clip, '_bg_clip') and sub_clip._bg_clip is not None:
                        clips.append(sub_clip._bg_clip)
                    clips.append(sub_clip)
                except Exception as e:
                    print(f"  Error creating subtitle: {e}")
        
        if len(clips) > 1:
            return CompositeVideoClip(clips, size=(config.VIDEO_WIDTH, config.VIDEO_HEIGHT))
        elif clips:
            # Single clip (video only)
            clip = clips[0]
            if not hasattr(clip, 'size') or clip.size != (config.VIDEO_WIDTH, config.VIDEO_HEIGHT):
                return CompositeVideoClip([clip], size=(config.VIDEO_WIDTH, config.VIDEO_HEIGHT))
            return clip
        
        return None

    def assemble_video(self, scenes: List[Scene], output_filename: str = None) -> str:
        """Assemble all scenes into a final video."""
        if not output_filename:
            timestamp = Path("shorts_").name
            from datetime import datetime
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_filename = f"shorts_{timestamp}.mp4"
        
        output_path = str(self.output_dir / output_filename)
        
        print(f"\nAssembling video with {len(scenes)} scenes...")
        
        # Create clips for each scene (start at 0; concatenation handles offsets)
        scene_clips = []
        for i, scene in enumerate(scenes):
            print(f"  Processing scene {i + 1}/{len(scenes)}...")
            clip = self._create_scene_clip(scene, 0)
            if clip:
                scene_clips.append(clip)
        
        if not scene_clips:
            raise RuntimeError("No scene clips could be created")
        
        # Calculate total duration
        total_duration = sum(s.duration for s in scenes)
        print(f"  Total duration: {total_duration:.1f}s")
        
        # Validate duration
        if total_duration < config.MIN_DURATION:
            print(f"  Warning: Duration {total_duration:.1f}s is below minimum ({config.MIN_DURATION}s)")
        elif total_duration > config.MAX_DURATION:
            print(f"  Warning: Duration {total_duration:.1f}s exceeds maximum ({config.MAX_DURATION}s)")
        
        # Clamp each scene clip to its declared duration
        concat_clips = []
        for scene, clip in zip(scenes, scene_clips):
            if hasattr(clip, 'duration') and clip.duration > scene.duration:
                clip = clip.subclipped(0, scene.duration)
            concat_clips.append(clip)
        
        if not concat_clips:
            raise RuntimeError("No clips to concatenate")
        
        # Crossfade transition between scenes (SCENE_TRANSITION seconds).
        # Scenes carry SCENE_AUDIO_PADDING of silent tail after the speech, so
        # the overlap never cuts audio: next scene fades in while previous one
        # fades out over that silent padding.
        trans = config.SCENE_TRANSITION
        if trans > 0 and len(concat_clips) > 1:
            faded = []
            for i, clip in enumerate(concat_clips):
                effects = []
                if i > 0:
                    effects.append(CrossFadeIn(trans))
                if i < len(concat_clips) - 1:
                    effects.append(CrossFadeOut(trans))
                faded.append(clip.with_effects(effects) if effects else clip)
            concat_clips = faded
            # Negative padding = overlap clips by `trans` seconds (compose mode)
            final_clip = concatenate_videoclips(
                concat_clips, method="compose", padding=-trans)
        else:
            # Concatenate clips
            final_clip = concatenate_videoclips(concat_clips, method="compose")
        
        # Ensure correct size
        if final_clip.size != (config.VIDEO_WIDTH, config.VIDEO_HEIGHT):
            final_clip = final_clip.resized(new_size=(config.VIDEO_WIDTH, config.VIDEO_HEIGHT))
        
        # Write final video
        print(f"  Rendering final video: {output_filename}")
        final_clip.write_videofile(
            output_path,
            fps=24,
            codec="libx264",
            audio_codec="aac",
            logger=None
        )
        
        # Clean up clips
        for clip in concat_clips:
            try:
                clip.close()
            except:
                pass
        final_clip.close()
        
        print(f"  Video saved: {output_path}")
        return output_path


if __name__ == "__main__":
    # Test the video assembler
    assembler = VideoAssembler()
    
    test_scenes = [
        Scene(
            id=1,
            text="Yapay zeka, makinelerin insan benzeri zekâ göstermesini sağlayan bir teknolojidir.",
            visual_description="AI technology",
            search_query="artificial intelligence",
            duration=8.0,
            image_path="",
            audio_path=""
        )
    ]
    
    # This would create a test video with black background
    print("Video assembler initialized successfully")


In [ ]:
import sys
from pathlib import Path

# Proje dosyaları üst hücrelerde /content içine yazıldı
sys.path.insert(0, "/content")

from IPython.display import Video, display

from pipeline.shorts_pipeline import ShortsPipeline




In [ ]:
# --- TTS engine selection at startup (Colab) ---
def _choose_tts_engine(service=None):
    import config
    provider = service or config.TTS_PROVIDER
    if provider:
        return provider
    if not sys.stdin.isatty():
        return "edge"
    print("\nSelect voice engine (free tier):")
    print("  [1] edge-tts    - free, best quality, needs network to services.google.com")
    print("  [2] cloudflare  - Workers AI (deepgram/aura-2-en), needs CLOUDFLARE_ACCOUNT_ID + CLOUDFLARE_API_TOKEN")
    print("  [3] ttsai       - tts.ai quality voices, 15k free chars/day, needs TTS_AI_API_KEY (get one free at tts.ai)")
    print("  [4] skip        - default (edge-tts)")
    try:
        choice = input("  Enter 1-4 (default 1): ").strip().lower()
    except (EOFError, KeyboardInterrupt):
        return "edge"
    mapping = {"1": "edge", "2": "cloudflare", "3": "ttsai"}
    return mapping.get(choice, "edge")


def generate(topic: str, voice_service: str = None) -> str:
    """Verilen konu ile Shorts videosu üretir. Çıktı yolunu döndürür."""
    if voice_service is None:
        voice_service = _choose_tts_engine()
    config.TTS_PROVIDER = voice_service
    print(f"Voice engine: {voice_service} ({config.TTS_PROVIDER})")
    pipeline = ShortsPipeline(voice_service=voice_service)
    output_path = pipeline.run(topic, voice_service=voice_service)
    path = Path(output_path)
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print()
        print(f"✅ Video hazır: {output_path} ({size_mb:.2f} MB)")
        display(Video(str(path), embed=False, width=360))
    else:
        print()
        print("❌ Video oluşturulamadı")
    return output_path

In [ ]:
# Konuyu değiştirip yeniden çalıştırın; ya da altına yeni hücre açıp generate("Yeni konu") yazın
output_path = generate("Game of Thrones dizisinin konusu nedir, neleri anlatır")


## İsteğe Bağlı: Videoyu Google Drive'a Kaydet

Aşağıdaki hücre, Drive varsa kopyalar; yoksa sessizce atlar.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/shorts_output")
    dest.mkdir(parents=True, exist_ok=True)
    if "output_path" in dir() and output_path and Path(output_path).exists():
        shutil.copy2(output_path, dest / Path(output_path).name)
        print(f"✓ Drive'a kopyalandı: {dest}")
    else:
        print("⚠️ Önce generate(...) çalıştırın")
except Exception as exc:
    print(f"Drive kullanılamadı (atlanıyor): {exc}")


## İpuçları

- **Yeni video:** Altına yeni bir Code hücresi açıp `generate("Yeni konu")` yazın.
- **Dosyalar:** Çıktılar `/content/output/` altında (sol panelde 📁 Files → klasörü yenileyin).
- **Sorun giderme:** Runtime → Restart runtime sonrası her şeyi yeniden çalıştırın (Run all).
- `services/colab_runner.py` bu deftere dahil değildir (Colab CLI yoluna özgüdür).
